M2 preprocessing (Tharsiga): downloads GloVe 100d, re-creates the split with Shevoni's make_or_load_split (seed 42) and checks its counts, builds vocab.json from train only, and tests ToxicDataset, collate_fn, build_dataloaders and load_glove_embeddings.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
EMB_DIR = '/content/drive/MyDrive/DL-Project/embeddings'
os.makedirs(EMB_DIR, exist_ok=True)

print("Drive mounted!")
print("Embeddings folder ready:", EMB_DIR)
print("Files in DL-Project:", os.listdir('/content/drive/MyDrive/DL-Project'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted!
Embeddings folder ready: /content/drive/MyDrive/DL-Project/embeddings
Files in DL-Project: ['data', 'results', 'embeddings']


In [9]:
!wget -c https://nlp.stanford.edu/data/glove.6B.zip -O /content/glove.6B.zip
!unzip -o /content/glove.6B.zip glove.6B.100d.txt -d /content/
!ls -lh /content/glove.6B.100d.txt

--2026-09-26 17:12:22--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-09-26 17:12:22--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

Archive:  /content/glove.6B.zip
  inflating: /content/glove.6B.100d.txt  
-rw-rw-r-- 1 root root 332M Aug  4  2014 /content/glove.6B.100d.txt


In [10]:
import shutil, os

src = '/content/glove.6B.100d.txt'
dst = '/content/drive/MyDrive/DL-Project/embeddings/glove.6B.100d.txt'
shutil.copy(src, dst)

print("Copied! Size in Drive:", round(os.path.getsize(dst) / 1e6), "MB")

# Quick health check
with open(dst, encoding='utf-8') as f:
    first = f.readline().split()
    n_lines = 1 + sum(1 for _ in f)

print("First word:", first[0])
print("Numbers per word:", len(first) - 1)
print("Total words:", n_lines)


Copied! Size in Drive: 347 MB
First word: the
Numbers per word: 100
Total words: 400000


In [11]:
!pip install -q iterative-stratification
%cd /content
!rm -rf shevoni
!git clone -q -b model/bilstm https://github.com/Vievek/DL-Project.git shevoni

# Load Shevoni's data_utils under a different name, so it won't clash with yours later
import importlib.util
spec = importlib.util.spec_from_file_location("shevoni_du", "/content/shevoni/src/data_utils.py")
sdu = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sdu)

SPLIT_DIR = '/content/drive/MyDrive/DL-Project/data/splits'
train_df, val_df, test_df = sdu.make_or_load_split(
    '/content/drive/MyDrive/DL-Project/data/train.csv',
    SPLIT_DIR,
    random_state=42
)

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name:5} rows={len(d):,}  labels={d[sdu.LABELS].sum().to_dict()}")

/content
Existing split found. Loading it.
train rows=111,699  labels={'toxic': 10706, 'severe_toxic': 1116, 'obscene': 5914, 'threat': 335, 'insult': 5514, 'identity_hate': 983}
val   rows=23,936  labels={'toxic': 2294, 'severe_toxic': 240, 'obscene': 1267, 'threat': 71, 'insult': 1182, 'identity_hate': 211}
test  rows=23,936  labels={'toxic': 2294, 'severe_toxic': 239, 'obscene': 1268, 'threat': 72, 'insult': 1181, 'identity_hate': 211}


In [12]:
%cd /content
!rm -rf DL-Project
!git clone -q -b model/textcnn https://github.com/Vievek/DL-Project.git
%cd /content/DL-Project

import sys; sys.path.insert(0, '/content/DL-Project')
from src import data_utils as du

cfg = du.load_config('config.yaml')
MAX_LEN = cfg['preprocessing']['max_length']
MIN_FREQ = cfg['preprocessing']['min_vocab_freq']

print(du.tokenize("You're an IDIOT!!\nSee http://x.com <b>now</b>"))

vocab = du.build_vocab(train_df['comment_text'], min_freq=MIN_FREQ)   # TRAIN only
print(f"Val OOV rate: {du.oov_rate(val_df['comment_text'], vocab):.2%}")

ids, length = du.encode(train_df['comment_text'].iloc[0], vocab, MAX_LEN)
print("Encoded length:", len(ids), "| real length:", length, "| first 20 ids:", ids[:20])

du.save_vocab(vocab, '/content/drive/MyDrive/DL-Project/data/splits/vocab.json')
print("Saved vocab.json to Drive")

/content
/content/DL-Project
["you're", 'an', 'idiot', '!', '!', 'see', 'now']
Unique tokens in train: 155,285
Vocab size (min_freq=2, incl. <pad>/<unk>): 74,916
Train token coverage: 99.12%
Top-20 tokens: [('.', 461999), ('the', 347362), (',', 332096), ('"', 273124), ('to', 208226), ('of', 157772), ('and', 156953), ('a', 151190), ('you', 144789), ('i', 143913), ('is', 123590), ('that', 108586), ('in', 102272), ('it', 91697), ('!', 79429), ('for', 71810), ('this', 68540), ('-', 66777), ('not', 65601), ('on', 62948)]
Val OOV rate: 1.74%
Encoded length: 128 | real length: 54 | first 20 ids: [708, 87, 3, 142, 145, 194, 38, 681, 5034, 12042, 1077, 98, 363, 29, 61, 2299, 9494, 4, 60, 6902]
Saved vocab.json to Drive


In [13]:
%cd /content
!rm -rf DL-Project
!git clone -q -b model/textcnn https://github.com/Vievek/DL-Project.git
%cd /content/DL-Project
import importlib; importlib.reload(du)

train_loader, val_loader, test_loader = du.build_dataloaders(cfg, train_df, val_df, test_df, vocab)
ids, lengths, mask, labels = next(iter(train_loader))

print("ids:", ids.shape, ids.dtype, "| lengths:", lengths.shape, "| mask:", mask.shape, mask.dtype, "| labels:", labels.shape, labels.dtype)
print("lengths   :", lengths[:8].tolist())
print("mask sums :", mask.sum(1)[:8].tolist())
print("batches   :", len(train_loader), len(val_loader), len(test_loader))

/content
/content/DL-Project
ids: torch.Size([32, 128]) torch.int64 | lengths: torch.Size([32]) | mask: torch.Size([32, 128]) torch.bool | labels: torch.Size([32, 6]) torch.float32
lengths   : [48, 6, 128, 91, 35, 22, 62, 128]
mask sums : [48, 6, 128, 91, 35, 22, 62, 128]
batches   : 3491 748 748


In [14]:
%cd /content
!rm -rf DL-Project
!git clone -q -b model/textcnn https://github.com/Vievek/DL-Project.git
%cd /content/DL-Project
import importlib; importlib.reload(du)

cfg = du.load_config('config.yaml')   # also checks the YAML indentation is valid
GLOVE = '/content/drive/MyDrive/DL-Project/embeddings/glove.6B.100d.txt'
emb = du.load_glove_embeddings(vocab, GLOVE, embed_dim=cfg['preprocessing']['embed_dim'])

print("shape:", emb.shape, emb.dtype)
print("pad row all zeros:", bool((emb[du.PAD_IDX] == 0).all()))
print("'the' first 5:", emb[vocab['the']][:5].tolist())

import torch.nn as nn
layer = nn.Embedding.from_pretrained(emb, freeze=False, padding_idx=du.PAD_IDX)
ids, lengths, mask, labels = next(iter(train_loader))
print("embedded batch:", layer(ids).shape)

/content
/content/DL-Project
GloVe: found 54,950 / 74,916 vocab words (73.35%); the rest are random-initialised
shape: torch.Size([74916, 100]) torch.float32
pad row all zeros: True
'the' first 5: [-0.03819400072097778, -0.24487000703811646, 0.7281200289726257, -0.3996100127696991, 0.08317200094461441]
embedded batch: torch.Size([32, 128, 100])
